In [1]:
from transformers import pipeline
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu

In [3]:
model_name = 'timpal0l/mdeberta-v3-base-squad2'
model_name = 'google-bert/bert-large-uncased-whole-word-masking-finetuned-squad'
model_name = 'deepset/roberta-large-squad2'



model_pipeline = pipeline(
   task='question-answering',
   model=model_name
)

Device set to use cpu


In [4]:
df_Q_a_A = pd.read_csv('data/Q_n_A_RAG.csv')

In [5]:
q = dict(df_Q_a_A.iloc[0]).get('Q')
c = dict(df_Q_a_A.iloc[0]).get('RAG')

In [6]:
df_Q_a_A['model_A_no_RAG'] = df_Q_a_A.apply(lambda row:model_pipeline(question=row['Q'],
                                                                      context=row['Q']) ,axis=1)

In [7]:
df_Q_a_A['model_A_with_RAG'] = df_Q_a_A.apply(lambda row:model_pipeline(question=row['Q'],
                                                                        context=row['RAG']) ,axis=1)

In [8]:
df_Q_a_A['model_A_no_RAG_'] = df_Q_a_A['model_A_no_RAG'].apply(lambda x: x.get('answer'))

In [9]:
df_Q_a_A['model_A_with_RAG_'] = df_Q_a_A['model_A_with_RAG'].apply(lambda x: x.get('answer').strip())

In [10]:
model_st = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [11]:
def get_score(str_1,str_2):
    try:
        score = sum(sum(cosine_similarity([model_st.encode(str_1)],
                                          [model_st.encode(str_2)])))
        return score
    except:
        print(str_1)
        print(str_2)
        print('\n')

In [12]:
df_Q_a_A['score'] = df_Q_a_A.apply(lambda row:get_score(str_1 = row['A'],
                                                        str_2 =row['model_A_with_RAG_']),
                          axis=1)

In [13]:
df_Q_a_A['score'].sum()/len(df_Q_a_A)

0.2873132405220531